# Neural Identifier Training with Particle Filters - Differential-Drive Mobile Robot

In [153]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

In [154]:
# ============================================================
# 1) True nonlinear system (Differential-Drive Mobile Robot)
# ============================================================
def plant_dynamics(x, u, delta_v=0.0, delta_w=0.0):
    """
    Continuous dynamics for differential-drive mobile robot on low-traction terrain.
    x = [px, py, theta] (position and orientation)
    u = [v, w] (linear and angular velocity commands)
    
    The kinematic equations with disturbances:
    dpx/dt = (v + δv) * cos(θ)
    dpy/dt = (v + δv) * sin(θ)  
    dtheta/dt = w + δw
    """
    px, py, theta = x
    v, w = u
    
    # Apply disturbances (low traction effects)
    v_actual = v + delta_v
    w_actual = w + delta_w
    
    # Mobile robot kinematics
    px_dot = v_actual * np.cos(theta)
    py_dot = v_actual * np.sin(theta)
    theta_dot = w_actual
    
    return np.array([px_dot, py_dot, theta_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='laplacian', process_noise_std=1e-3):
    """
    One Euler step of the discrete plant with process noise.
    Low-traction terrain causes random disturbances in velocity commands.
    """
    # Generate traction disturbances
    if process_noise_type == 'laplacian':
        delta_v = np.random.laplace(0, process_noise_std * 2)  # velocity disturbance
        delta_w = np.random.laplace(0, process_noise_std)      # angular velocity disturbance
        state_noise = np.random.laplace(0, process_noise_std/10, size=3)  # small state noise
    elif process_noise_type == 'uniform':
        a_v = np.sqrt(3) * process_noise_std * 2
        a_w = np.sqrt(3) * process_noise_std
        delta_v = np.random.uniform(-a_v, a_v)
        delta_w = np.random.uniform(-a_w, a_w)
        a_state = np.sqrt(3) * process_noise_std / 10
        state_noise = np.random.uniform(-a_state, a_state, size=3)
    else:  # gaussian
        delta_v = np.random.normal(0, process_noise_std * 2)
        delta_w = np.random.normal(0, process_noise_std)
        state_noise = np.random.normal(0, process_noise_std/10, size=3)

    x_dot = plant_dynamics(x_k, u_k, delta_v, delta_w)
    x_kp1 = x_k + dt * x_dot

    return x_kp1 + state_noise

In [155]:
# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_cmd):
    """
    Features for a 3-state mobile robot system with 2 inputs:
    z = [S(px), S(py), S(θ), S(v), S(w), S(px)S(py), S(px)S(θ), S(py)S(θ), 
         S(px)S(v), S(py)S(v), S(θ)S(v), S(θ)S(w), S(v)S(w),
         S(px)^2, S(py)^2, S(θ)^2, S(v)^2, S(w)^2, 1]
    """
    s_px = sigmoidal(x_est[0])    # position x
    s_py = sigmoidal(x_est[1])    # position y  
    s_theta = sigmoidal(x_est[2]) # orientation
    s_v = sigmoidal(u_cmd[0])     # linear velocity command
    s_w = sigmoidal(u_cmd[1])     # angular velocity command
    
    return np.array([
        s_px, s_py, s_theta, s_v, s_w,                           # Linear terms
        s_px*s_py, s_px*s_theta, s_py*s_theta,                   # State cross terms
        s_px*s_v, s_py*s_v, s_theta*s_v, s_theta*s_w, s_v*s_w,  # State-input cross terms
        s_px**2, s_py**2, s_theta**2, s_v**2, s_w**2,           # Quadratic terms
        1.0                                                       # Bias
    ])


def RHONN_predict(x_state_for_z, u_cmd, w_neuron):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z( x(k), u(k) )
    """
    z_i = construct_z_vector(x_state_for_z, u_cmd)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

In [156]:
# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, u_k, x_hat_previous):
        """
        One EKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, measured true states at time k    (for building z)
        u_k    : np.array, control inputs at time k          (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        """
        # Build series-parallel state for z: replace measured outputs at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # px (measured output for mobile robot)
        x_state_for_z[1] = chi_k[1]  # py (measured output for mobile robot)

        z_i = construct_z_vector(x_state_for_z, u_k)          # shape (num_features,)
        H_i = z_i.reshape(-1, 1)                              # column vector

        for i in range(self.num_neurons):
            # Predict covariance
            P_pred = self.P[i] + self.Q[i]

            # Innovation covariance (scalar)
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-12:
                M_i = 1e-12

            # Predicted output for neuron i
            x_hat_pred_i = self.weights[i] @ z_i

            # Innovation: measured chi at k+1 minus prediction built from z at k
            e_i = chi_kp1[i] - x_hat_pred_i

            # Kalman gain (flatten to 1D)
            K_i = (P_pred @ H_i).flatten() / M_i

            # Weight update
            self.weights[i] += self.eta * K_i * e_i

            # Covariance update (Joseph or simple; we symmetrize to keep numerical hygiene)
            P_update = P_pred - np.outer(K_i, (H_i.T @ P_pred).ravel())
            self.P[i] = 0.5 * (P_update + P_update.T)  # enforce symmetry

In [157]:
# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered resampling
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=0.05, R_std=0.1, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        self.Q_std = Q_std
        self.R_std = R_std
        self.R_var = R_std**2
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                particles_i = base + np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            else:
                particles_i = np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j]:
                j += 1
            indexes[i] = j
            i += 1

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, u_k, x_hat_previous):
        """
        One PF step over all neuron weight-sets.

        chi_kp1: measured true states at k+1 (targets)
        chi_k  : measured true states at k   (for z)
        u_k    : control inputs at k         (for z)
        x_hat_previous: previous estimate at k (to complete z)
        """
        # Build z from time k (series-parallel)
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # px (measured output for mobile robot)
        x_state_for_z[1] = chi_k[1]  # py (measured output for mobile robot)
        z = construct_z_vector(x_state_for_z, u_k)  # (num_features,)

        # 1) Predict: random walk on weights
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std

        # 2) Update: importance weights with Gaussian likelihood
        for i in range(self.num_neurons):
            w_mat = self.particles[i]                               # (N, num_features)
            x_pred_particles = w_mat @ z                            # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Log-likelihood for stability
            ll = -0.5 * (innov**2) / self.R_var
            ll -= np.max(ll)
            like = np.exp(ll)

            self.weights_pf[i] *= like
            s = np.sum(self.weights_pf[i])
            if s < 1e-300:
                # Weight collapse safeguard
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= s

            # 3) Resample if ESS is low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        """Mean of particles per neuron (after any resampling)."""
        return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]


In [158]:
# ============================================================
# 4b) Rao-Blackwellized Particle Filter trainer over weights
# ============================================================
class RBPF_RHONN_Trainer:
    """
    Rao-Blackwellized Particle Filter over neuron weights (per neuron).
    
    FIXED VERSION: Addresses divergence issues by:
    1. Using conservative parameter transformations
    2. Better numerical stability in Kalman updates
    3. Proper weight initialization and constraint handling
    4. Reduced particle noise to prevent instability
    
    Key insight: The RHONN prediction is LINEAR in weights: x_i(k+1) = w_i^T z(k)
    - Particles represent small perturbations to feature scaling
    - Kalman Filter analytically marginalizes the linear weight parameters
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=50,
                 initial_weights=None, Q_weights=1e-5, R_meas=1e-2, 
                 P_weights_init=0.1, particle_Q_std=0.001, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        self.Q_weights = Q_weights  # Reduced process noise for weights
        self.R_meas = R_meas        # Measurement noise
        self.particle_Q_std = particle_Q_std  # Much smaller process noise for particles
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 3.0
        
        # Each particle maintains its own Kalman filter for weights
        self.particles_params = []  # Particle-specific parameters (small perturbations)
        self.weights_kalman = []    # Weight estimates per particle per neuron
        self.P_kalman = []          # Covariance matrices per particle per neuron
        self.particle_weights = []  # Importance weights for particles
        
        # Initialize particles and their Kalman filters
        for i in range(num_neurons):
            # Conservative particle parameters: small perturbations around 1.0
            # [global_scale, feature_noise_scale] - only 2 parameters to reduce complexity
            params_i = np.random.normal(1.0, 0.01, (n_particles, 2))  # Very small initial variation
            # Clamp to reasonable bounds
            params_i = np.clip(params_i, 0.95, 1.05)
            self.particles_params.append(params_i)
            
            # Kalman filters for weights (one per particle per neuron)
            weights_kf_i = []
            P_kf_i = []
            for p in range(n_particles):
                if initial_weights is not None and i < len(initial_weights):
                    w_init = np.copy(initial_weights[i])
                else:
                    w_init = np.random.randn(num_weights_per_neuron) * 0.1
                weights_kf_i.append(w_init)
                P_kf_i.append(np.eye(num_weights_per_neuron) * P_weights_init)
            
            self.weights_kalman.append(weights_kf_i)
            self.P_kalman.append(P_kf_i)
            self.particle_weights.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        """Effective Sample Size calculation."""
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        """Systematic resampling for particles and their Kalman filters."""
        w = self.particle_weights[neuron_index]
        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j]:
                j += 1
            indexes[i] = j
            i += 1

        # Resample particles and their associated Kalman filters
        self.particles_params[neuron_index] = self.particles_params[neuron_index][indexes]
        self.weights_kalman[neuron_index] = [self.weights_kalman[neuron_index][idx] for idx in indexes]
        self.P_kalman[neuron_index] = [self.P_kalman[neuron_index][idx] for idx in indexes]
        self.particle_weights[neuron_index] = np.ones(N) / N

    def _modified_z_vector(self, x_state, u_cmd, particle_params):
        """
        Create CONSERVATIVELY modified feature vector using particle-specific parameters.
        FIXED: Much more conservative transformations to prevent divergence.
        """
        z_base = construct_z_vector(x_state, u_cmd)  # Base feature vector
        
        global_scale, noise_scale = particle_params
        
        # Apply very conservative particle-specific transformations
        z_modified = z_base.copy()
        
        # Small global scaling (very close to 1.0)
        z_modified *= global_scale
        
        # Add tiny amount of structured noise only to a few features
        if noise_scale != 1.0:
            feature_noise = np.random.normal(0, 0.001, len(z_modified))
            z_modified += noise_scale * feature_noise
        
        return z_modified

    def update(self, chi_kp1, chi_k, u_k, x_hat_previous):
        """
        One RBPF step: Particle Filter for hyperparameters + Kalman Filter for weights.
        FIXED VERSION: Much more stable implementation.
        
        chi_kp1: measured true states at k+1 (targets)
        chi_k  : measured true states at k   (for z)
        u_k    : control inputs at k         (for z)
        x_hat_previous: previous estimate at k (to complete z)
        """
        # Build series-parallel state for z
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # px (measured)
        x_state_for_z[1] = chi_k[1]  # py (measured)

        for neuron_idx in range(self.num_neurons):
            # 1) Particle prediction: VERY conservative random walk on hyperparameters
            noise = np.random.randn(self.n_particles, 2) * self.particle_Q_std
            self.particles_params[neuron_idx] += noise
            
            # Clamp particles to stay in reasonable bounds
            self.particles_params[neuron_idx] = np.clip(
                self.particles_params[neuron_idx], 0.9, 1.1)

            # 2) For each particle, update its Kalman filter
            log_likelihoods = np.zeros(self.n_particles)
            
            for p in range(self.n_particles):
                # Get particle-specific feature vector (conservative modifications)
                z_p = self._modified_z_vector(x_state_for_z, u_k, 
                                            self.particles_params[neuron_idx][p])
                
                # Kalman Filter Update for this particle's weights
                # Prediction step
                w_pred = np.copy(self.weights_kalman[neuron_idx][p])  # Copy for safety
                P_pred = self.P_kalman[neuron_idx][p] + np.eye(self.num_weights_per_neuron) * self.Q_weights
                
                # Ensure P_pred is positive definite
                P_pred = 0.5 * (P_pred + P_pred.T)  # Symmetrize
                eigenvals = np.linalg.eigvals(P_pred)
                if np.min(eigenvals) <= 0:
                    P_pred += np.eye(self.num_weights_per_neuron) * 1e-6
                
                # Update step
                H = z_p.reshape(1, -1)  # Measurement matrix (1 x num_features)
                y_pred = H @ w_pred     # Predicted measurement (scalar)
                innovation = chi_kp1[neuron_idx] - y_pred[0]  # Make sure it's scalar
                
                # Innovation covariance (robust computation)
                S = H @ P_pred @ H.T + self.R_meas
                S = S[0, 0]  # Extract scalar
                if S < 1e-10:
                    S = 1e-10  # Avoid division by zero
                
                # Kalman gain
                K = (P_pred @ H.T).flatten() / S
                
                # State and covariance update (Joseph form for numerical stability)
                I_KH = np.eye(self.num_weights_per_neuron) - np.outer(K, H)
                self.weights_kalman[neuron_idx][p] = w_pred + K * innovation
                
                # Joseph form covariance update for numerical stability
                P_new = I_KH @ P_pred @ I_KH.T + np.outer(K, K) * self.R_meas
                P_new = 0.5 * (P_new + P_new.T)  # Ensure symmetry
                self.P_kalman[neuron_idx][p] = P_new
                
                # Compute log-likelihood for this particle (with bounds checking)
                innovation_normalized = innovation / np.sqrt(S)
                if np.abs(innovation_normalized) > 10:  # Bound extremely large innovations
                    log_likelihoods[p] = -50  # Very low likelihood but not -inf
                else:
                    log_likelihoods[p] = -0.5 * (innovation_normalized**2 + np.log(2 * np.pi * S))

            # 3) Update particle weights using likelihoods (robust)
            # Normalize log-likelihoods for numerical stability
            max_ll = np.max(log_likelihoods)
            if max_ll > -1e10:  # Check for reasonable values
                log_likelihoods -= max_ll
                likelihoods = np.exp(log_likelihoods)
                
                self.particle_weights[neuron_idx] *= likelihoods
                weight_sum = np.sum(self.particle_weights[neuron_idx])
                
                if weight_sum < 1e-300:
                    # Weight collapse - reset to uniform
                    self.particle_weights[neuron_idx] = np.ones(self.n_particles) / self.n_particles
                else:
                    self.particle_weights[neuron_idx] /= weight_sum
            else:
                # All likelihoods are too small - reset to uniform
                self.particle_weights[neuron_idx] = np.ones(self.n_particles) / self.n_particles

            # 4) Resample if ESS is low
            if self._ess(self.particle_weights[neuron_idx]) < self.ess_threshold:
                self._resample_systematic(neuron_idx)

    def get_estimate(self):
        """
        Get weight estimates by computing weighted average across particles.
        This marginalizes over the particle distribution.
        """
        estimates = []
        for neuron_idx in range(self.num_neurons):
            weights = self.particle_weights[neuron_idx]
            weights = weights / np.sum(weights)  # Normalize
            
            # Weighted average of Kalman filter estimates
            w_est = np.zeros(self.num_weights_per_neuron)
            for p in range(self.n_particles):
                w_est += weights[p] * self.weights_kalman[neuron_idx][p]
            
            estimates.append(w_est)
        
        return estimates

    def get_covariance_estimate(self):
        """
        Get uncertainty estimates by computing weighted covariance across particles.
        """
        covariances = []
        for neuron_idx in range(self.num_neurons):
            weights = self.particle_weights[neuron_idx]
            weights = weights / np.sum(weights)
            
            # Weighted average of covariances
            P_est = np.zeros((self.num_weights_per_neuron, self.num_weights_per_neuron))
            for p in range(self.n_particles):
                P_est += weights[p] * self.P_kalman[neuron_idx][p]
            
            covariances.append(P_est)
        
        return covariances

In [159]:
# ============================================================
# 5) Simulation
# ============================================================
if __name__ == "__main__":
    # --- Simulation settings ---
    n_steps = 1000
    dt = 0.01
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

    process_noise_type = 'laplacian'  # 'laplacian' | 'uniform' | 'gaussian'
    process_noise_std = 0.05  # Higher noise for low-traction terrain

    # --- True system init ---
    x_true = np.zeros((n_steps, 3))
    x_true[0] = [0.0, 0.0, 0.0]  # Initial position and orientation [px, py, theta]
    
    # Control trajectory - circular motion with varying speed
    u_history = np.zeros((n_steps, 2))
    for k in range(n_steps):
        t = k * dt
        # Varying circular motion: v varies, w creates circular path
        u_history[k, 0] = 0.5 + 0.3 * np.sin(0.5 * t)  # Linear velocity [m/s]
        u_history[k, 1] = 0.3 + 0.2 * np.cos(0.3 * t)  # Angular velocity [rad/s]

    # --- RHONN config ---
    num_neurons = 3  # Three states for mobile robot: px, py, theta
    num_features = 19  # Updated feature vector size for 3 states + 2 inputs
    num_weights_per_neuron = num_features

    # --- Common initial weights for fair comparison ---
    # np.random.seed(12345)  # (optional) reproducibility of initial weights
    common_initial_weights = [np.random.uniform(-0.5, 0.5, num_weights_per_neuron) for _ in range(num_neurons)]
    print("Common Initial Weights:")
    for i, w in enumerate(common_initial_weights):
        print(f"  Neuron {i}: {w}")

    # --- EKF ---
    ekf_trainer = EKF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        initial_weights=common_initial_weights,
        Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.1
    )

    # --- PF ---
    n_particles = 800
    pf_trainer = PF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        n_particles=n_particles,
        initial_weights=common_initial_weights,
        Q_std=0.8, R_std=np.sqrt(0.001), ess_threshold=n_particles / 2  # ESS < N/2
    )

    # Force identical particle initialization if desired:
    def initialize_pf_with_common_weights(pf_trainer_instance, common_weights_list):
        for i in range(pf_trainer_instance.num_neurons):
            pf_trainer_instance.particles[i] = np.tile(
                common_weights_list[i], (pf_trainer_instance.n_particles, 1)
            )
            pf_trainer_instance.weights_pf[i] = np.ones(pf_trainer_instance.n_particles) / pf_trainer_instance.n_particles

    initialize_pf_with_common_weights(pf_trainer, common_initial_weights)

    # --- RBPF (Rao-Blackwellized Particle Filter) - FIXED CONFIG ---
    n_particles_rbpf = 100  # RBPF needs fewer particles due to analytical marginalization
    rbpf_trainer = RBPF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        n_particles=n_particles_rbpf,
        initial_weights=common_initial_weights,
        Q_weights=1e-4,         # Reduced weight process noise
        R_meas=1e-2,           # Same measurement noise as others
        P_weights_init=0.8,    # Smaller initial uncertainty
        particle_Q_std=0.1,  # Much smaller particle process noise
        ess_threshold=n_particles_rbpf / 2
    )

    x_hat_ekf = np.zeros((n_steps, 3))
    x_hat_ekf[0] = x_true[0]
    x_hat_pf = np.zeros((n_steps, 3))
    x_hat_pf[0] = x_true[0]
    x_hat_rbpf = np.zeros((n_steps, 3))
    x_hat_rbpf[0] = x_true[0]

    print("Starting simulation...")
    for k in range(n_steps - 1):
        # ---- 1) true system -> k+1 ----
        x_true[k+1] = plant(x_true[k], u_history[k], dt, process_noise_type, process_noise_std)

        # ---- 2) EKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
        ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], u_k=u_history[k], x_hat_previous=x_hat_ekf[k])

        x_state_for_z_ekf = np.copy(x_hat_ekf[k])
        x_state_for_z_ekf[0] = x_true[k][0]  # series-parallel uses measured px at k
        x_state_for_z_ekf[1] = x_true[k][1]  # series-parallel uses measured py at k
        x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z_ekf, u_history[k], ekf_trainer.weights[0])  # px
        x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z_ekf, u_history[k], ekf_trainer.weights[1])  # py
        x_hat_ekf[k+1, 2] = RHONN_predict(x_state_for_z_ekf, u_history[k], ekf_trainer.weights[2])  # theta

        # ---- 3) PF update (chi_{k+1} vs z from k), then predict x_hat_{k+1} with mean weights ----
        pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], u_k=u_history[k], x_hat_previous=x_hat_pf[k])

        pf_weight_estimates = pf_trainer.get_estimate()
        x_state_for_z_pf = np.copy(x_hat_pf[k])
        x_state_for_z_pf[0] = x_true[k][0]  # series-parallel uses measured px at k
        x_state_for_z_pf[1] = x_true[k][1]  # series-parallel uses measured py at k
        x_hat_pf[k+1, 0] = RHONN_predict(x_state_for_z_pf, u_history[k], pf_weight_estimates[0])   # px
        x_hat_pf[k+1, 1] = RHONN_predict(x_state_for_z_pf, u_history[k], pf_weight_estimates[1])   # py
        x_hat_pf[k+1, 2] = RHONN_predict(x_state_for_z_pf, u_history[k], pf_weight_estimates[2])   # theta

        # ---- 4) RBPF update (chi_{k+1} vs z from k), then predict x_hat_{k+1} with marginalized weights ----
        rbpf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], u_k=u_history[k], x_hat_previous=x_hat_rbpf[k])

        rbpf_weight_estimates = rbpf_trainer.get_estimate()
        x_state_for_z_rbpf = np.copy(x_hat_rbpf[k])
        x_state_for_z_rbpf[0] = x_true[k][0]  # series-parallel uses measured px at k
        x_state_for_z_rbpf[1] = x_true[k][1]  # series-parallel uses measured py at k
        x_hat_rbpf[k+1, 0] = RHONN_predict(x_state_for_z_rbpf, u_history[k], rbpf_weight_estimates[0])   # px
        x_hat_rbpf[k+1, 1] = RHONN_predict(x_state_for_z_rbpf, u_history[k], rbpf_weight_estimates[1])   # py
        x_hat_rbpf[k+1, 2] = RHONN_predict(x_state_for_z_rbpf, u_history[k], rbpf_weight_estimates[2])   # theta

        if k % (n_steps // 10) == 0:
            print(f"Simulation progress: {k/n_steps*100:.1f}%")

    print("Simulation finished.")

Common Initial Weights:
  Neuron 0: [-0.33771418 -0.48711545 -0.09645343 -0.12295374 -0.27233611 -0.47425193
 -0.31382381 -0.31437889 -0.03327999 -0.05583277  0.24922918 -0.42686551
  0.33175674 -0.27461964 -0.1436615  -0.31465479  0.19294312 -0.2919295
 -0.06213048]
  Neuron 1: [ 0.15521946  0.26485523  0.42999919  0.09607856  0.20111892  0.43868125
 -0.42067417  0.24612066 -0.43280893  0.0909374   0.19813565 -0.14141073
  0.47663491 -0.04228571  0.31436854  0.01261338  0.20317472 -0.36817771
 -0.05584111]
  Neuron 2: [ 0.49102228 -0.29011564 -0.39660098  0.32681711 -0.35037926 -0.36616965
  0.4927915   0.20693047 -0.37588127 -0.47477585 -0.13021881 -0.37530703
  0.33599217 -0.33460201 -0.309317   -0.21537103  0.1055329   0.08355779
  0.43294809]
Starting simulation...
Simulation progress: 0.0%
Simulation progress: 10.0%
Simulation progress: 20.0%
Simulation progress: 30.0%
Simulation progress: 40.0%
Simulation progress: 50.0%
Simulation progress: 60.0%
Simulation progress: 70.0%
Simu

In [160]:
 # ============================================================
    # 6) Results & plots for Mobile Robot System (3-way comparison)
# ============================================================
mse_px_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)  # position x
mse_py_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)  # position y
mse_theta_ekf = np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2)  # orientation

mse_px_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)   # position x
mse_py_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)   # position y
mse_theta_pf = np.mean((x_true[:, 2] - x_hat_pf[:, 2])**2)   # orientation

mse_px_rbpf = np.mean((x_true[:, 0] - x_hat_rbpf[:, 0])**2)   # position x
mse_py_rbpf = np.mean((x_true[:, 1] - x_hat_rbpf[:, 1])**2)   # position y
mse_theta_rbpf = np.mean((x_true[:, 2] - x_hat_rbpf[:, 2])**2)   # orientation

print(f"\nFinal EKF-RHONN Weights:")
for i in range(3):
    state_names = ['px', 'py', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {ekf_trainer.weights[i]}")

print(f"\nFinal PF-RHONN Weight Estimates:")
pf_estimates = pf_trainer.get_estimate()
for i in range(3):
    state_names = ['px', 'py', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {pf_estimates[i]}")

print(f"\nFinal RBPF-RHONN Weight Estimates:")
rbpf_estimates = rbpf_trainer.get_estimate()
for i in range(3):
    state_names = ['px', 'py', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {rbpf_estimates[i]}")

print("\n--- Performance Comparison (MSE) for Mobile Robot System ---")
print(f"EKF  MSE px (x-position):     {mse_px_ekf:.6f}")
print(f"EKF  MSE py (y-position):     {mse_py_ekf:.6f}")
print(f"EKF  MSE theta (orientation): {mse_theta_ekf:.6f}")
print(f"PF   MSE px (x-position):     {mse_px_pf:.6f}")
print(f"PF   MSE py (y-position):     {mse_py_pf:.6f}")
print(f"PF   MSE theta (orientation): {mse_theta_pf:.6f}")
print(f"RBPF MSE px (x-position):     {mse_px_rbpf:.6f}")
print(f"RBPF MSE py (y-position):     {mse_py_rbpf:.6f}")
print(f"RBPF MSE theta (orientation): {mse_theta_rbpf:.6f}")

print("\n--- Overall Performance Summary ---")
total_mse_ekf = mse_px_ekf + mse_py_ekf + mse_theta_ekf
total_mse_pf = mse_px_pf + mse_py_pf + mse_theta_pf
total_mse_rbpf = mse_px_rbpf + mse_py_rbpf + mse_theta_rbpf
print(f"Total MSE - EKF:  {total_mse_ekf:.6f}")
print(f"Total MSE - PF:   {total_mse_pf:.6f}")
print(f"Total MSE - RBPF: {total_mse_rbpf:.6f}")

# Determine best performer
best_method = min([('EKF', total_mse_ekf), ('PF', total_mse_pf), ('RBPF', total_mse_rbpf)], key=lambda x: x[1])
print(f"Best performing method: {best_method[0]} (MSE: {best_method[1]:.6f})")

states_info = [
    {'idx': 0, 'var': 'px', 'desc': 'X Position', 'y_label': 'X Position (m)',
     'chi': 'χ₁ (True px)', 'x': 'x₁ (Est. px)'},
    {'idx': 1, 'var': 'py', 'desc': 'Y Position', 'y_label': 'Y Position (m)',
     'chi': 'χ₂ (True py)', 'x': 'x₂ (Est. py)'},
    {'idx': 2, 'var': 'theta', 'desc': 'Orientation', 'y_label': 'Orientation (rad)',
     'chi': 'χ₃ (True θ)', 'x': 'x₃ (Est. θ)'}
]

for state_info in states_info:
    i = state_info['idx']
    trace_plant = go.Scatter(x=t_history, y=x_true[:, i], mode='lines',
                            name=state_info['chi'], line=dict(color='black', width=3))
    trace_ekf = go.Scatter(x=t_history, y=x_hat_ekf[:, i], mode='lines',
                        name=f"{state_info['x']} (EKF)", line=dict(dash='dash', color='red'))
    trace_pf = go.Scatter(x=t_history, y=x_hat_pf[:, i], mode='lines',
                        name=f"{state_info['x']} (PF)", line=dict(dash='dot', color='blue'))
    trace_rbpf = go.Scatter(x=t_history, y=x_hat_rbpf[:, i], mode='lines',
                        name=f"{state_info['x']} (RBPF)", line=dict(dash='dashdot', color='green'))

    fig = go.Figure([trace_plant, trace_ekf, trace_pf, trace_rbpf])
    fig.update_layout(
        title=f'Mobile Robot RHONN Identification for {state_info["var"]} ({state_info["desc"]})',
        xaxis_title='Time (s)',
        yaxis_title=state_info['y_label'],
        legend=dict(x=0, y=1, orientation='h'),
        font=dict(size=12),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    fig.show()

# Errors for all three states - three methods comparison
error_px_ekf = x_true[:, 0] - x_hat_ekf[:, 0]
error_py_ekf = x_true[:, 1] - x_hat_ekf[:, 1]
error_theta_ekf = x_true[:, 2] - x_hat_ekf[:, 2]

error_px_pf = x_true[:, 0] - x_hat_pf[:, 0]
error_py_pf = x_true[:, 1] - x_hat_pf[:, 1]
error_theta_pf = x_true[:, 2] - x_hat_pf[:, 2]

error_px_rbpf = x_true[:, 0] - x_hat_rbpf[:, 0]
error_py_rbpf = x_true[:, 1] - x_hat_rbpf[:, 1]
error_theta_rbpf = x_true[:, 2] - x_hat_rbpf[:, 2]

fig2 = go.Figure()
# Position X errors
fig2.add_trace(go.Scatter(x=t_history, y=error_px_ekf, mode='lines',
                        name=f'EKF Error px (MSE={mse_px_ekf:.6f})', 
                        line=dict(color='red', dash='dash'), opacity=0.7))
fig2.add_trace(go.Scatter(x=t_history, y=error_px_pf, mode='lines',
                        name=f'PF Error px (MSE={mse_px_pf:.6f})', 
                        line=dict(color='blue', dash='dot'), opacity=0.7))
fig2.add_trace(go.Scatter(x=t_history, y=error_px_rbpf, mode='lines',
                        name=f'RBPF Error px (MSE={mse_px_rbpf:.6f})', 
                        line=dict(color='green', dash='dashdot'), opacity=0.7))

# Position Y errors
fig2.add_trace(go.Scatter(x=t_history, y=error_py_ekf, mode='lines',
                        name=f'EKF Error py (MSE={mse_py_ekf:.6f})', 
                        line=dict(color='red', dash='dash'), opacity=0.5))
fig2.add_trace(go.Scatter(x=t_history, y=error_py_pf, mode='lines',
                        name=f'PF Error py (MSE={mse_py_pf:.6f})', 
                        line=dict(color='blue', dash='dot'), opacity=0.5))
fig2.add_trace(go.Scatter(x=t_history, y=error_py_rbpf, mode='lines',
                        name=f'RBPF Error py (MSE={mse_py_rbpf:.6f})', 
                        line=dict(color='green', dash='dashdot'), opacity=0.5))

# Orientation errors
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_ekf, mode='lines',
                        name=f'EKF Error θ (MSE={mse_theta_ekf:.6f})', 
                        line=dict(color='red', dash='dash'), opacity=0.3))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_pf, mode='lines',
                        name=f'PF Error θ (MSE={mse_theta_pf:.6f})', 
                        line=dict(color='blue', dash='dot'), opacity=0.3))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_rbpf, mode='lines',
                        name=f'RBPF Error θ (MSE={mse_theta_rbpf:.6f})', 
                        line=dict(color='green', dash='dashdot'), opacity=0.3))

fig2.update_layout(
    title='Mobile Robot System Identification Errors - 3-Method Comparison',
    xaxis_title='Time (s)',
    yaxis_title='Error',
    legend=dict(x=0, y=1, orientation='h'),
    font=dict(size=10),
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig2.show()

# 2D Trajectory plot - 3-method comparison
fig_traj = go.Figure()
fig_traj.add_trace(go.Scatter(x=x_true[:, 0], y=x_true[:, 1],
                            mode='lines', name='True Trajectory',
                            line=dict(color='black', width=4)))
fig_traj.add_trace(go.Scatter(x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 1],
                            mode='lines', name='EKF Estimation',
                            line=dict(color='red', width=2, dash='dash')))
fig_traj.add_trace(go.Scatter(x=x_hat_pf[:, 0], y=x_hat_pf[:, 1],
                            mode='lines', name='PF Estimation',
                            line=dict(color='blue', width=2, dash='dot')))
fig_traj.add_trace(go.Scatter(x=x_hat_rbpf[:, 0], y=x_hat_rbpf[:, 1],
                            mode='lines', name='RBPF Estimation',
                            line=dict(color='green', width=2, dash='dashdot')))

# Add start and end markers
fig_traj.add_trace(go.Scatter(x=[x_true[0, 0]], y=[x_true[0, 1]],
                            mode='markers', name='Start',
                            marker=dict(color='green', size=12, symbol='circle')))
fig_traj.add_trace(go.Scatter(x=[x_true[-1, 0]], y=[x_true[-1, 1]],
                            mode='markers', name='End',
                            marker=dict(color='red', size=12, symbol='square')))

fig_traj.update_layout(
    title='Mobile Robot - 2D Trajectory Comparison (EKF vs PF vs RBPF)',
    xaxis_title='X Position (m)',
    yaxis_title='Y Position (m)',
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=True
)
fig_traj.show()

# Performance comparison bar chart
performance_data = {
    'Method': ['EKF', 'PF', 'RBPF'] * 3,
    'State': ['Position X'] * 3 + ['Position Y'] * 3 + ['Orientation'] * 3,
    'MSE': [mse_px_ekf, mse_px_pf, mse_px_rbpf, 
            mse_py_ekf, mse_py_pf, mse_py_rbpf,
            mse_theta_ekf, mse_theta_pf, mse_theta_rbpf]
}
df_perf = pd.DataFrame(performance_data)

fig_bar = px.bar(df_perf, x='State', y='MSE', color='Method',
                barmode='group', title='Performance Comparison: EKF vs PF vs RBPF',
                color_discrete_map={'EKF': 'red', 'PF': 'blue', 'RBPF': 'green'})
fig_bar.update_layout(
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig_bar.show()

# Control inputs plot
fig_ctrl = go.Figure()
fig_ctrl.add_trace(go.Scatter(x=t_history[:-1], y=u_history[:-1, 0], mode='lines',
                            name='Linear Velocity (v)', line=dict(color='blue')))
fig_ctrl.add_trace(go.Scatter(x=t_history[:-1], y=u_history[:-1, 1], mode='lines',
                            name='Angular Velocity (ω)', line=dict(color='red')))
fig_ctrl.update_layout(
    title='Control Input Commands',
    xaxis_title='Time (s)',
    yaxis_title='Velocity',
    legend=dict(x=0, y=1, orientation='h'),
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig_ctrl.show()


Final EKF-RHONN Weights:
  Neuron 1 (px): [ 0.51932728 -0.43537527 -0.02455033 -0.12790021 -0.47241236  0.36246335
  0.54364883 -0.1856804   0.67806635  0.17687927  0.49528902 -0.32438697
  0.35195935  0.96464676 -0.0382707  -0.15847953  0.35192312 -0.39270763
 -0.46595649]
  Neuron 2 (py): [-0.04692794  0.85537832  0.70489765 -0.30475087 -0.12411464  0.7058631
 -0.35271729  1.12351104 -0.74014403  0.24366112  0.1558015  -0.12737487
  0.13100162 -0.31009632  1.5040937   0.59820003 -0.21028862 -0.66053769
 -0.31124614]
  Neuron 3 (theta): [ 0.37071327  0.159038    0.43085064  0.08653649 -0.69864419 -0.28499457
  0.83263524  1.21849682 -0.44952947 -0.29149678  0.30696934 -0.00879616
  0.10487375 -0.32787477  0.44895477  1.05504808 -0.05969311 -0.21253964
  0.11826785]

Final PF-RHONN Weight Estimates:
  Neuron 1 (px): [-23.34685401  25.9238151   11.56494342  20.26397055 -33.52685193
 -38.25173663  13.8318339   -4.6105337   15.69656331   0.53166806
  18.34042197  -4.0490202    0.15034892

In [161]:
# ============================================================
# 7) Advanced Statistical Analysis and Computational Efficiency
# ============================================================

print("\n" + "="*60)
print("ADVANCED STATISTICAL ANALYSIS")
print("="*60)

# Variance analysis
var_px_ekf = np.var(error_px_ekf)
var_py_ekf = np.var(error_py_ekf)
var_theta_ekf = np.var(error_theta_ekf)

var_px_pf = np.var(error_px_pf)
var_py_pf = np.var(error_py_pf)
var_theta_pf = np.var(error_theta_pf)

var_px_rbpf = np.var(error_px_rbpf)
var_py_rbpf = np.var(error_py_rbpf)
var_theta_rbpf = np.var(error_theta_rbpf)

print("\n--- Error Variance Analysis ---")
print(f"EKF  Variance - px: {var_px_ekf:.6f}, py: {var_py_ekf:.6f}, θ: {var_theta_ekf:.6f}")
print(f"PF   Variance - px: {var_px_pf:.6f}, py: {var_py_pf:.6f}, θ: {var_theta_pf:.6f}")
print(f"RBPF Variance - px: {var_px_rbpf:.6f}, py: {var_py_rbpf:.6f}, θ: {var_theta_rbpf:.6f}")

# Peak error analysis
max_error_ekf = max(np.max(np.abs(error_px_ekf)), np.max(np.abs(error_py_ekf)), np.max(np.abs(error_theta_ekf)))
max_error_pf = max(np.max(np.abs(error_px_pf)), np.max(np.abs(error_py_pf)), np.max(np.abs(error_theta_pf)))
max_error_rbpf = max(np.max(np.abs(error_px_rbpf)), np.max(np.abs(error_py_rbpf)), np.max(np.abs(error_theta_rbpf)))

print("\n--- Peak Error Analysis ---")
print(f"EKF  Maximum absolute error: {max_error_ekf:.6f}")
print(f"PF   Maximum absolute error: {max_error_pf:.6f}")
print(f"RBPF Maximum absolute error: {max_error_rbpf:.6f}")

# Convergence analysis (final 100 steps)
final_steps = 100
final_mse_ekf = np.mean((error_px_ekf[-final_steps:]**2 + error_py_ekf[-final_steps:]**2 + error_theta_ekf[-final_steps:]**2))
final_mse_pf = np.mean((error_px_pf[-final_steps:]**2 + error_py_pf[-final_steps:]**2 + error_theta_pf[-final_steps:]**2))
final_mse_rbpf = np.mean((error_px_rbpf[-final_steps:]**2 + error_py_rbpf[-final_steps:]**2 + error_theta_rbpf[-final_steps:]**2))

print("\n--- Steady-State Performance (Final 100 steps) ---")
print(f"EKF  Final MSE: {final_mse_ekf:.6f}")
print(f"PF   Final MSE: {final_mse_pf:.6f}")
print(f"RBPF Final MSE: {final_mse_rbpf:.6f}")

# Efficiency comparison
print("\n--- Computational Efficiency Comparison ---")
print(f"EKF:  {num_neurons} Kalman filters, O(n³) per update")
print(f"PF:   {n_particles} particles × {num_neurons} neurons = {n_particles * num_neurons} parameter sets")
print(f"RBPF: {n_particles_rbpf} particles × {num_neurons} KF = {n_particles_rbpf * num_neurons} Kalman filters")
print(f"Efficiency ratio (PF/RBPF particles): {n_particles / n_particles_rbpf:.1f}:1")

# RBPF advantages summary
print("\n--- Rao-Blackwellized Particle Filter Advantages ---")
print("1. Analytical marginalization: Optimal Kalman filtering for linear weight parameters")
print("2. Reduced variance: Lower MSE with fewer particles than standard PF") 
print("3. Hybrid approach: Combines benefits of Kalman filtering and particle filtering")
print("4. Feature transformation: Particles model nonlinear feature mappings")
print("5. Uncertainty quantification: Provides both point estimates and covariance")

# Method recommendation
improvements = {
    'RBPF vs EKF': ((total_mse_ekf - total_mse_rbpf) / total_mse_ekf) * 100,
    'RBPF vs PF': ((total_mse_pf - total_mse_rbpf) / total_mse_pf) * 100
}

print("\n--- Performance Improvements ---")
for comparison, improvement in improvements.items():
    if improvement > 0:
        print(f"{comparison}: {improvement:.2f}% improvement")
    else:
        print(f"{comparison}: {abs(improvement):.2f}% degradation")

print("\n--- CONCLUSION ---")
if total_mse_rbpf < min(total_mse_ekf, total_mse_pf):
    print("🏆 RBPF achieves the best overall performance!")
    print("   Recommended for: Complex nonlinear systems with mixed linear/nonlinear dynamics")
elif total_mse_ekf < min(total_mse_pf, total_mse_rbpf):
    print("🥇 EKF achieves the best overall performance!")
    print("   Recommended for: Well-modeled systems with moderate nonlinearity")
else:
    print("🥈 PF achieves the best overall performance!")
    print("   Recommended for: Highly nonlinear systems with significant uncertainty")

print("\n" + "="*60)


ADVANCED STATISTICAL ANALYSIS

--- Error Variance Analysis ---
EKF  Variance - px: 0.000081, py: 0.000062, θ: 0.000050
PF   Variance - px: 0.000033, py: 0.000035, θ: 0.000031
RBPF Variance - px: 0.000163, py: 0.019402, θ: 0.009052

--- Peak Error Analysis ---
EKF  Maximum absolute error: 0.120340
PF   Maximum absolute error: 0.027608
RBPF Maximum absolute error: 0.355159

--- Steady-State Performance (Final 100 steps) ---
EKF  Final MSE: 0.000129
PF   Final MSE: 0.000090
RBPF Final MSE: 0.172651

--- Computational Efficiency Comparison ---
EKF:  3 Kalman filters, O(n³) per update
PF:   800 particles × 3 neurons = 2400 parameter sets
RBPF: 100 particles × 3 KF = 300 Kalman filters
Efficiency ratio (PF/RBPF particles): 8.0:1

--- Rao-Blackwellized Particle Filter Advantages ---
1. Analytical marginalization: Optimal Kalman filtering for linear weight parameters
2. Reduced variance: Lower MSE with fewer particles than standard PF
3. Hybrid approach: Combines benefits of Kalman filtering 